In [ ]:
import torch
import math


class LinearLayer:
    def __init__(self,in_features,out_features):
        kaim = math.sqrt(2/in_features)
        self.weights = torch.randn(out_features,in_features)*kaim
        self.bias = torch.zeros(out_features)


    def forward(self,x):
        self.x = x
        return x @ self.weights.T + self.bias

    def backward(self,grad_out):
        grad_inputs = grad_out @ self.weights
        x_flat = self.x.reshape(-1, self.x.shape[-1])
        grad_out_flat = grad_out.reshape(-1, grad_out.shape[-1])
        self.weights.grad = grad_out_flat.T @ x_flat
        self.bias.grad = grad_out_flat.sum(dim=0)
        
        return grad_inputs


class Relu:
    def forward(self,x):
        self.x = x
        return torch.clamp(x,min=0)

    def backward(self,grad_out):
        return grad_out * (self.x > 0)


class MyMlp:
    def __init__(self,in_features,hidden_features,out_features):
        self.linear1 = LinearLayer(in_features,hidden_features)
        self.relu = Relu()
        self.linear2 = LinearLayer(hidden_features,out_features)

    def forward(self,x):
        x = self.linear1.forward(x)
        x = self.relu.forward(x)
        x = self.linear2.forward(x)
        
        return x

    def backward(self,grad_out):
        grad_linear2 = self.linear2.backward(grad_out)
        grad_relu = self.relu.backward(grad_linear2)
        grad_linear1 = self.linear1.backward(grad_relu)

        return grad_linear1

    def parameters(self):
        return [self.linear1.weights,self.linear1.bias,self.linear2.weights,self.linear2.bias]

class MyMseLoss:
    
    def forward(self, y_pred, y_true):
        self.y_pred = y_pred
        self.y_true = y_true
        return torch.mean((y_pred - y_true)**2)

    def backward(self):
        N = self.y_pred.numel()
        grad_out = (2 / N) * (self.y_pred - self.y_true)
        return grad_out


class MyOptim:

    def __init__(self,params,lr):
        self.params = list(params)
        self.lr = lr

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()

    def step(self):
        for p in self.params:
            with torch.no_grad():
                if p.grad is not None:
                    p-=self.lr*p.grad
            
                
in_features = 100
out_features = 10
hidden_features = 16
lr = 0.0001
B = 32
X = torch.randn(B,in_features)
y_true = torch.randn(B,out_features)

model = MyMlp(in_features,hidden_features,out_features)
params = list(model.parameters())
loss_fn = MyMseLoss()
optim = MyOptim(params,lr)

for epoch in range(100):

    optim.zero_grad()

    # forward
    y_pred = model.forward(X)

    # loss
    loss = loss_fn.forward(y_pred,y_true)

    # backward
    grad_out = loss_fn.backward()
    model.backward(grad_out)

    optim.step()

    if epoch % 10 == 0:
         print(f"loss at epoch {epoch} is {loss}")
